# Tutorial: Ejercicio 3.15 en AMPL - construccion del barco for dummies

Audiencia:
- Personas que quieren ver el problema del barco escrito formalmente en AMPL.
- Personas que quieren entender como modelar precedencias y decisiones binarias.

Objetivos:
- Traducir el problema a sets, parametros y variables.
- Resolver el modelo con `amplpy`.
- Leer una solucion clara y comparable con Python.


## Enunciado del ejercicio

La empresa Monkey Island debe terminar la construccion de un barco en `15` dias.

Cada una de las `9` tareas se puede hacer:
- en modo normal,
- o en modo rapido.

El modo rapido cuesta dinero, pero reduce la duracion.
Ademas, existen precedencias entre tareas.

La capacidad es ilimitada, asi que varias tareas pueden ejecutarse en paralelo si las precedencias lo permiten.

Pregunta:
**¿Que tareas conviene hacer en modo rapido para cumplir el plazo al minimo costo?**


## Mapa del notebook

1. Ver los datos.
2. Entender la decision binaria.
3. Escribir el modelo AMPL.
4. Resolverlo.
5. Reconstruir un cronograma temprano en Python.
6. Probar una variante.


In [7]:
TAREAS = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]
MODOS = [1, 2]
PRECEDENCIAS = [("A", "C"), ("A", "D"), ("B", "E"), ("C", "E"), ("E", "F"), ("D", "G"), ("F", "H"), ("G", "I")]
DURACION = {
    ("A", 1): 3, ("A", 2): 1,
    ("B", 1): 4, ("B", 2): 2,
    ("C", 1): 2, ("C", 2): 0.5,
    ("D", 1): 5, ("D", 2): 2,
    ("E", 1): 6, ("E", 2): 1,
    ("F", 1): 2, ("F", 2): 1,
    ("G", 1): 4, ("G", 2): 3,
    ("H", 1): 3, ("H", 2): 2,
    ("I", 1): 5, ("I", 2): 4,
}
COSTO_RAPIDO = {"A": 4, "B": 1, "C": 1, "D": 1, "E": 3, "F": 7, "G": 9, "H": 5, "I": 8}
PREDECESORES = {
    "A": [],
    "B": [],
    "C": ["A"],
    "D": ["A"],
    "E": ["B", "C"],
    "F": ["E"],
    "G": ["D"],
    "H": ["F"],
    "I": ["G"],
}
FECHA_LIMITE = 15
ESPERADO = {"costo_total": 2, "tareas_rapidas": {"C": 1, "D": 1}, "duracion_total": 15}

TAREAS


['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']

## Paso 1 - Que decide el modelo

La decision principal es binaria:
- `Rapida[i] = 1` si la tarea `i` se hace en modo rapido,
- `Rapida[i] = 0` si la tarea `i` se hace en modo normal.

Tambien necesitamos una variable continua `Inicio[i]` para indicar cuando empieza cada tarea.


In [8]:
piezas = [
    {"pieza": "set TAREAS", "significado": "las actividades A..I"},
    {"pieza": "set PRECEDENCIAS", "significado": "pares (i, k) donde i debe terminar antes de que k empiece"},
    {"pieza": "param duracion", "significado": "duracion por tarea y modo"},
    {"pieza": "param costo_rapido", "significado": "costo extra de hacer una tarea en rapido"},
    {"pieza": "var Inicio", "significado": "instante de inicio de cada tarea"},
    {"pieza": "var Rapida", "significado": "decision binaria normal/rapida"},
]

piezas


[{'pieza': 'set TAREAS', 'significado': 'las actividades A..I'},
 {'pieza': 'set PRECEDENCIAS',
  'significado': 'pares (i, k) donde i debe terminar antes de que k empiece'},
 {'pieza': 'param duracion', 'significado': 'duracion por tarea y modo'},
 {'pieza': 'param costo_rapido',
  'significado': 'costo extra de hacer una tarea en rapido'},
 {'pieza': 'var Inicio', 'significado': 'instante de inicio de cada tarea'},
 {'pieza': 'var Rapida', 'significado': 'decision binaria normal/rapida'}]

## Paso 2 - Preparar AMPL desde Python

Si tu kernel aun no tiene `amplpy`, primero instala:

```python
%pip install amplpy
```

En este modelo la parte clave es la restriccion de precedencia:
- una tarea hija no puede empezar antes de que termine su predecesora.


In [9]:
from amplpy import ampl_notebook


def crear_ampl():
    ampl = ampl_notebook(modules=["highs"], license_uuid="default")
    ampl.option["solver"] = "highs"
    ampl.option["solver_msg"] = 0
    return ampl


def leer_variable_1d(variable, claves):
    bruto = variable.get_values().to_dict()
    limpio = {}
    for clave in claves:
        if clave in bruto:
            valor = bruto[clave]
        else:
            valor = bruto[(clave,)]
        limpio[clave] = float(valor)
    return limpio


def calcular_cronograma_temprano(rapida):
    inicio = {}
    fin = {}
    duracion = {}
    for tarea in TAREAS:
        duracion[tarea] = DURACION[(tarea, 2)] if rapida[tarea] else DURACION[(tarea, 1)]
        if not PREDECESORES[tarea]:
            inicio[tarea] = 0
        else:
            inicio[tarea] = max(fin[pre] for pre in PREDECESORES[tarea])
        fin[tarea] = inicio[tarea] + duracion[tarea]
    return {
        "inicio": inicio,
        "fin": fin,
        "duracion": duracion,
        "duracion_total": max(fin.values()),
    }


MODELO_AMPL = r'''
set TAREAS ordered;
set MODOS ordered;
set PRECEDENCIAS within {TAREAS, TAREAS};

param due_date >= 0;
param duracion {TAREAS, MODOS} >= 0;
param costo_rapido {TAREAS} >= 0;

var Inicio {TAREAS} >= 0;
var Rapida {TAREAS} binary;

minimize CostoTotal:
    sum {i in TAREAS} costo_rapido[i] * Rapida[i];

subject to FechaLimite {i in TAREAS}:
    Inicio[i] + duracion[i,1] * (1 - Rapida[i]) + duracion[i,2] * Rapida[i] <= due_date;

subject to Precedencia {(i,k) in PRECEDENCIAS}:
    Inicio[k] >= Inicio[i] + duracion[i,1] * (1 - Rapida[i]) + duracion[i,2] * Rapida[i];
'''


## Paso 3 - Ver el modelo AMPL completo

Antes de resolver, vale la pena leer el modelo entero.
Asi se hace mucho mas evidente donde esta:
- la funcion objetivo,
- la fecha limite,
- y la logica de precedencias.


In [10]:
print(MODELO_AMPL)



set TAREAS ordered;
set MODOS ordered;
set PRECEDENCIAS within {TAREAS, TAREAS};

param due_date >= 0;
param duracion {TAREAS, MODOS} >= 0;
param costo_rapido {TAREAS} >= 0;

var Inicio {TAREAS} >= 0;
var Rapida {TAREAS} binary;

minimize CostoTotal:
    sum {i in TAREAS} costo_rapido[i] * Rapida[i];

subject to FechaLimite {i in TAREAS}:
    Inicio[i] + duracion[i,1] * (1 - Rapida[i]) + duracion[i,2] * Rapida[i] <= due_date;

subject to Precedencia {(i,k) in PRECEDENCIAS}:
    Inicio[k] >= Inicio[i] + duracion[i,1] * (1 - Rapida[i]) + duracion[i,2] * Rapida[i];



## Paso 4 - Resolver el modelo

AMPL puede devolver una solucion optima con muchas horas de inicio validas.
Para que el resultado sea mas facil de leer, despues de obtener la decision binaria usamos Python para reconstruir el cronograma mas temprano posible.


In [11]:
def resolver_barco_ampl(fecha_limite=FECHA_LIMITE):
    ampl = crear_ampl()
    ampl.eval(MODELO_AMPL)

    ampl.set["TAREAS"] = TAREAS
    ampl.set["MODOS"] = MODOS
    ampl.set["PRECEDENCIAS"] = PRECEDENCIAS
    ampl.param["due_date"] = fecha_limite
    ampl.param["duracion"] = DURACION
    ampl.param["costo_rapido"] = COSTO_RAPIDO
    ampl.solve()

    rapida_raw = leer_variable_1d(ampl.var["Rapida"], TAREAS)
    rapida = {t: int(round(rapida_raw[t])) for t in TAREAS}
    cronograma = calcular_cronograma_temprano(rapida)

    resultado = {
        "costo_total": int(round(float(ampl.obj["CostoTotal"].value()))),
        "rapida": rapida,
        "inicio": cronograma["inicio"],
        "fin": cronograma["fin"],
        "duracion_total": cronograma["duracion_total"],
    }
    return ampl, resultado


ampl_base, resultado_base = resolver_barco_ampl()
assert resultado_base["costo_total"] == ESPERADO["costo_total"]
assert {t: v for t, v in resultado_base["rapida"].items() if v} == ESPERADO["tareas_rapidas"]
assert resultado_base["duracion_total"] == ESPERADO["duracion_total"]

print("Salida nativa de AMPL:")
print(ampl_base.get_output("display Rapida, Inicio;"))

cronograma = []
for tarea in TAREAS:
    cronograma.append(
        {
            "tarea": tarea,
            "rapida": resultado_base["rapida"][tarea],
            "inicio": resultado_base["inicio"][tarea],
            "fin": resultado_base["fin"][tarea],
        }
    )

cronograma


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: Salida nativa de AMPL:
: Rapida Inicio    :=
A    0       0
B    0       0
C    1       3
D    1       3
E    0       4
F    0      10
G    0       5
H    0      12
I    0      10
;




[{'tarea': 'A', 'rapida': 0, 'inicio': 0, 'fin': 3},
 {'tarea': 'B', 'rapida': 0, 'inicio': 0, 'fin': 4},
 {'tarea': 'C', 'rapida': 1, 'inicio': 3, 'fin': 3.5},
 {'tarea': 'D', 'rapida': 1, 'inicio': 3, 'fin': 5},
 {'tarea': 'E', 'rapida': 0, 'inicio': 4, 'fin': 10},
 {'tarea': 'F', 'rapida': 0, 'inicio': 10, 'fin': 12},
 {'tarea': 'G', 'rapida': 0, 'inicio': 5, 'fin': 9},
 {'tarea': 'H', 'rapida': 0, 'inicio': 12, 'fin': 15},
 {'tarea': 'I', 'rapida': 0, 'inicio': 9, 'fin': 14}]

## Paso 5 - Que significa la solucion

La respuesta importante no es el cronograma exacto de AMPL, sino la decision binaria:
- acelerar `C` y `D`,
- no acelerar las otras,
- pagar un costo total de `2`.

Luego, con esa decision, el cronograma temprano termina justo en `15` dias.


## Ejercicio guiado

Prueba ahora con plazo `14` dias.

Pregunta previa:
**si el plazo baja en un solo dia, cambia mucho la combinacion optima de tareas rapidas?**


In [12]:
_, resultado_due_14 = resolver_barco_ampl(fecha_limite=14)

{
    "costo_con_due_15": resultado_base["costo_total"],
    "tareas_rapidas_con_due_15": {t: v for t, v in resultado_base["rapida"].items() if v},
    "costo_con_due_14": resultado_due_14["costo_total"],
    "tareas_rapidas_con_due_14": {t: v for t, v in resultado_due_14["rapida"].items() if v},
}


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: 

{'costo_con_due_15': 2,
 'tareas_rapidas_con_due_15': {'C': 1, 'D': 1},
 'costo_con_due_14': 4,
 'tareas_rapidas_con_due_14': {'D': 1, 'E': 1}}

## Comparacion con el libro

El libro reporta que la solucion optima acelera `C` y `D`, con costo total `2`, para cumplir el plazo de `15` dias.

En AMPL la informacion clave que debe coincidir es:
- el costo minimo,
- las tareas elegidas en modo rapido,
- y el cumplimiento del plazo.

Los tiempos de inicio exactos pueden variar entre cronogramas optimos equivalentes.


In [13]:
comparacion_libro = {
    "costo_coincide": resultado_base["costo_total"] == ESPERADO["costo_total"],
    "tareas_rapidas_coinciden": {t: v for t, v in resultado_base["rapida"].items() if v} == ESPERADO["tareas_rapidas"],
    "duracion_total_coincide": resultado_base["duracion_total"] == ESPERADO["duracion_total"],
    "conclusion": "La respuesta coincide con la del libro en costo y tareas aceleradas.",
}

comparacion_libro


{'costo_coincide': True,
 'tareas_rapidas_coinciden': True,
 'duracion_total_coincide': True,
 'conclusion': 'La respuesta coincide con la del libro en costo y tareas aceleradas.'}